# Data Exploration
在这个项目中，我们有不同的数据集：
1. 纯文本数据，用于继续预训练，给通用模型添加额外信息，知识。
2. 高质量指令-回答对（问答数据，对话数据），让模型学习指令跟随和对话格式，提高多轮对话能力。
3. 偏好数据集，对于一个问题，保存好答案vs坏答案，优化模型的回答质量，学习人类偏好。

## 1. 继续预训练
核心目的
- 领域知识注入：Domain Knowledge Injection
- 向通用大模型注入医疗专业知识，让模型理解医疗术语，病症，治疗方法等。
- 需要高质量，大规模的医疗文本数据


In [1]:
import os
import json
with open('../data/pretrain/fever.txt', 'r') as f:
    pt_data = f.read()

In [2]:
print(pt_data[:500])

第一章论
传染病是指由病原微生物，如朊粒、病毒、衣原体、立克次体、支原体（mycoplasma)细菌真菌、螺旋体和寄生虫，如原虫、蠕虫、医学昆虫感染人体后产生的有传染性、在一定条件下可造成流行的疾病。感染性疾病是指由病原体感染所致的疾病，包括传染病和非传染性感染性疾病。
传染病学是一门研究各种传染病在人体内外发生、发展、传播、诊断、治疗和预防规律的学科。重点研究各种传染病的发病机制、临床表现、诊断和治疗方法，同时兼顾流行病学和预防措施的研究，做到防治结合。
传染病学与其他学科有密切联系，其基础学科和相关学科包括病原生物学、分子生物学、免疫学、人体寄生虫学、流行病学、病理学、药理学和诊断学等。掌握这些学科的基本知识、基本理论和基本技能对学好传染病学起着非常重要的作用。
在人类历史长河中，传染病不仅威胁着人类的健康和生命，而且影响着人类文明的进程，甚至改写过人类历史。人类在与传染病较量过程中，取得了许多重大战果，19世纪以来，病原微生物的不断发现及其分子生物学的兴起，推动了生命科学乃至整个医学的发展；疫苗的研究诞生了感染免疫学，奠定了免疫学的理论基础，已用来研究各种疾病的发生机制及防治手段


## 2. SFT / Instruction Tuning: 指令微调
- 让模型学会理解和遵循人类的指令
- 从语言模型， next token prediction，转变为任务执行。
- 学习多轮对话格式，保持上下文理解和回答的连贯性。

In [3]:
with open('../data/finetune/medical_sft_1K_format.jsonl', 'r') as f:
    lines = f.readlines()
ft_data = [json.loads(line) for line in lines]

In [13]:
conversations = ft_data[10]['conversations']
for turn in conversations:
    print(f"{turn['from']}: {turn['value']}")

human: 小儿髓母细胞瘤的就诊科室是什么？
gpt: 肿瘤科；肿瘤外科


## 3. 偏好对齐
- SFT阶段只是让模型学习如何回答，以及指令跟随，而并没有学习到怎么回答的好
- 无法区分回答质量的优劣
- 可能生成看似正确但是有害的内容

所以我们需要继续调整模型，让模型生成更准确的回答
1. 训练Reward model，然后PPO
2. 直接DPO / GRPO

In [14]:
with open('../data/reward/dpo_zh_500.jsonl', 'r') as f:
    lines = f.readlines()
dpo_data = [json.loads(line) for line in lines]

In [20]:
dpo_data[2]

{'system': '你是一个非常聪明的AI助手，非常擅长按照指示行事。尽你所能地帮助。',
 'history': [],
 'question': '请为以下电影情节提供一个电影标题：1964年的一个简短场景中，年迈肥胖的意大利裔美国人杰克·拉莫塔（罗伯特·德尼罗饰）练习喜剧表演。然后整部电影发生在回忆中。1941年，拉莫塔在一场重要的拳击比赛中对阵吉米·里夫斯，他在比赛中首次失利。杰克的兄弟乔伊·拉莫塔（乔·佩西饰）与他的黑手党联系人之一萨尔维·巴茨（弗兰克·文森特饰）讨论了一个有可能争夺中量级冠军头衔的机会。此后不久，杰克在布朗克斯社区的一个露天游泳池里看到一个名叫维基（凯西·莫里亚蒂饰）的15岁女孩。尽管他已经结婚，但最终他与她开始了一段关系。1943年，杰克击败了糖·雷·罗宾逊，并在三周后进行了一场复赛。尽管杰克在比赛中占据上风，但裁判出人意料地判罗宾逊获胜，乔伊认为罗宾逊只是因为下周要入伍到美国陆军才赢得了比赛。到了1947年，杰克与维基结婚。杰克经常担心维基对其他男人有感情，尤其是当她对杰克下一场比赛的对手托尼·贾尼罗做出不经意的评论时。当他在当地黑帮老大汤米·科莫（尼古拉斯·科拉桑托饰）和维基面前残忍地打败贾尼罗时，他的嫉妒心显而易见。当乔伊在科帕卡巴纳与记者们讨论胜利时，他被看到维基走向萨尔维和他的团队的桌子而分心。乔伊与维基交谈，维基说她已经对他的兄弟失望了。乔伊责怪萨尔维，激烈地袭击他，打斗甚至蔓延到俱乐部外。科莫后来命令他们道歉，并让乔伊告诉杰克，如果他想有机会争夺由科莫控制的冠军头衔，他必须先认输。在与比利·福克斯的比赛中，杰克在短暂地殴打对手后，甚至都不费力地进行抵抗。他随后因涉嫌故意输掉比赛而被暂时停赛，尽管他意识到自己判断错误时已经太迟。最终他被恢复资格，并在1949年赢得了...',
 'response_chosen': '"愤怒的传奇：杰克·拉莫塔的故事"',
 'response_rejected': '根据您提供的电影情节，这里有一些潜在的电影标题想法：\n\n1. "布朗克斯的公牛：爱情、失落和皮革的故事"\n2. "愤怒的公牛：杰克·拉莫塔的未被揭示的故事"\n3. "爱与拳拳：杰克·拉莫塔的故事"\n4. "中量级黑帮分子：拳击剧"\n5. "科帕卡巴纳团队：拳击、背叛和兄弟之爱的故事"\n6. "拳击手的心：嫉妒

In [23]:
sample = dpo_data[2]
for key in sample.keys():
    print(f'{key}: {sample.get(key)}')

system: 你是一个非常聪明的AI助手，非常擅长按照指示行事。尽你所能地帮助。
history: []
question: 请为以下电影情节提供一个电影标题：1964年的一个简短场景中，年迈肥胖的意大利裔美国人杰克·拉莫塔（罗伯特·德尼罗饰）练习喜剧表演。然后整部电影发生在回忆中。1941年，拉莫塔在一场重要的拳击比赛中对阵吉米·里夫斯，他在比赛中首次失利。杰克的兄弟乔伊·拉莫塔（乔·佩西饰）与他的黑手党联系人之一萨尔维·巴茨（弗兰克·文森特饰）讨论了一个有可能争夺中量级冠军头衔的机会。此后不久，杰克在布朗克斯社区的一个露天游泳池里看到一个名叫维基（凯西·莫里亚蒂饰）的15岁女孩。尽管他已经结婚，但最终他与她开始了一段关系。1943年，杰克击败了糖·雷·罗宾逊，并在三周后进行了一场复赛。尽管杰克在比赛中占据上风，但裁判出人意料地判罗宾逊获胜，乔伊认为罗宾逊只是因为下周要入伍到美国陆军才赢得了比赛。到了1947年，杰克与维基结婚。杰克经常担心维基对其他男人有感情，尤其是当她对杰克下一场比赛的对手托尼·贾尼罗做出不经意的评论时。当他在当地黑帮老大汤米·科莫（尼古拉斯·科拉桑托饰）和维基面前残忍地打败贾尼罗时，他的嫉妒心显而易见。当乔伊在科帕卡巴纳与记者们讨论胜利时，他被看到维基走向萨尔维和他的团队的桌子而分心。乔伊与维基交谈，维基说她已经对他的兄弟失望了。乔伊责怪萨尔维，激烈地袭击他，打斗甚至蔓延到俱乐部外。科莫后来命令他们道歉，并让乔伊告诉杰克，如果他想有机会争夺由科莫控制的冠军头衔，他必须先认输。在与比利·福克斯的比赛中，杰克在短暂地殴打对手后，甚至都不费力地进行抵抗。他随后因涉嫌故意输掉比赛而被暂时停赛，尽管他意识到自己判断错误时已经太迟。最终他被恢复资格，并在1949年赢得了...
response_chosen: "愤怒的传奇：杰克·拉莫塔的故事"
response_rejected: 根据您提供的电影情节，这里有一些潜在的电影标题想法：

1. "布朗克斯的公牛：爱情、失落和皮革的故事"
2. "愤怒的公牛：杰克·拉莫塔的未被揭示的故事"
3. "爱与拳拳：杰克·拉莫塔的故事"
4. "中量级黑帮分子：拳击剧"
5. "科帕卡巴纳团队：拳击、背叛和兄弟之爱的故事"
6. "拳击手的心：嫉妒、贪婪和救赎的故事"
7. "游泳池的罗密欧：拳击手的爱情与失落的